# GDELT News Headlines — WTI Crude Oil

This notebook queries the GDELT Project API to retrieve news article headlines 
related to WTI crude oil prices over a two-year period (2024–2026). Articles are 
downloaded week by week to avoid API rate limiting. The raw dataset is then cleaned 
by filtering to English-language articles only, removing duplicates, and parsing 
timestamps. The cleaned dataset contains article titles, publication timestamps, 
source domains, and URLs, and is saved to 
`01_data/processed/gdelt_wti_clean.csv`.

### General imports

In [4]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
import sqlite3

### Main search function

Adjusts request headers so that instead of identifying as Python, requests are sent with a Mozilla browser User-Agent string. This avoids being blocked by GDELT's bot detection.

**Note:** GDELT API results are capped at 250 articles per request. Weeks where the download returns exactly 250 articles may be truncated. This affects a small number of high-activity periods and is considered acceptable for the purposes of this study.

In [8]:
def search_gdelt(query, start_date, end_date, max_records=250):
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {
        "query": query,
        "mode": "artlist",
        "maxrecords": max_records,
        "startdatetime": start_date.strftime("%Y%m%d%H%M%S"),
        "enddatetime": end_date.strftime("%Y%m%d%H%M%S"),
        "format": "json"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers, timeout=30)
    return response.json()

### Download by date range

Executes the search function iteratively over a range of weeks between `start_date` and `end_date`. A cooldown between requests is included to reduce the number of rejections caused by GDELT's rate limiting.

In [9]:
def download_gdelt_range(query, start_date, end_date, sleep_seconds=8):
    all_articles = []
    current = start_date
    
    while current < end_date:
        week_end = min(current + timedelta(days=7), end_date)
        try:
            results = search_gdelt(query, current, week_end)
            articles = results.get("articles", [])
            for a in articles:
                a['week_start'] = current.strftime("%Y-%m-%d")
            all_articles.extend(articles)
            print(f"{current.date()} → {week_end.date()}: {len(articles)} articles")
        except Exception as e:
            print(f"{current.date()} → {week_end.date()}: ERROR — {str(e)[:50]}")
        
        current = week_end
        time.sleep(sleep_seconds)
    
    return pd.DataFrame(all_articles)

### Queries
These is what GDELT will look for.

In [10]:
# Celda — Additional queries
additional_queries = [
    # Oil related
    "crude oil market",
    "OPEC production cut",
    "oil supply demand",
    "petroleum inventory",
    "Brent crude price",
    "oil inventory report",
    "energy market outlook",
    "crude oil WTI price",

    # New high-signal additions
    "Iran sanctions oil",
    "Saudi Arabia oil production",
    "China oil demand",
    "Russia oil exports",
    "oil supply disruption",
]


### Download weekly headlines

GDELT does not provide article body text through its API — only metadata including title, URL, publication date, domain, and language. Body text is retrieved separately in `04_gdelt_scraper.ipynb` by scraping each article URL individually.

In [11]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

# Load existing URLs from DB
existing_urls = set(pd.read_sql("SELECT url FROM articles", conn)['url'].tolist())
print(f"Existing articles in DB: {len(existing_urls)}")

dfs = []
for query in additional_queries:
    print(f"\nDownloading: '{query}'")
    df_query = download_gdelt_range(
        query=query,
        start_date=datetime(2026, 1, 1),
        end_date=datetime(2026, 5, 12),
        sleep_seconds=12
    )
    print(f"  -> {len(df_query)} articles")
    dfs.append(df_query)

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal before dedup: {len(df_all)}")

# Deduplicate by URL
df_all = df_all.drop_duplicates(subset='url').reset_index(drop=True)
print(f"After dedup: {len(df_all)}")

# Keep only new articles not already in DB
df_new = df_all[~df_all['url'].isin(existing_urls)].reset_index(drop=True)
print(f"New articles to add: {len(df_new)}")

conn.close()

Existing articles in DB: 16326

Downloading: 'crude oil market'
2026-01-01 → 2026-01-08: ERROR — Expecting value: line 1 column 1 (char 0)
2026-01-08 → 2026-01-15: ERROR — Expecting value: line 1 column 1 (char 0)
2026-01-15 → 2026-01-22: ERROR — Expecting value: line 1 column 1 (char 0)
2026-01-22 → 2026-01-29: 250 articles
2026-01-29 → 2026-02-05: 250 articles
2026-02-05 → 2026-02-12: 250 articles
2026-02-12 → 2026-02-19: 250 articles
2026-02-19 → 2026-02-26: 250 articles
2026-02-26 → 2026-03-05: 250 articles
2026-03-05 → 2026-03-12: ERROR — Expecting value: line 1 column 1 (char 0)
2026-03-12 → 2026-03-19: ERROR — Expecting value: line 1 column 1 (char 0)
2026-03-19 → 2026-03-26: ERROR — Expecting value: line 1 column 1 (char 0)
2026-03-26 → 2026-04-02: 250 articles
2026-04-02 → 2026-04-09: ERROR — Expecting value: line 1 column 1 (char 0)
2026-04-09 → 2026-04-16: 250 articles
2026-04-16 → 2026-04-23: 250 articles
2026-04-23 → 2026-04-30: 250 articles
2026-04-30 → 2026-05-07: ERROR 

In [12]:
import os

# Deduplicate new downloads by URL among themselves
df_all = df_all.drop_duplicates(subset='url').reset_index(drop=True)
print(f"New articles after dedup: {len(df_all)}")
print(f"\nLanguages:\n{df_all['language'].value_counts().head()}")
print(f"\nTop domains:\n{df_all['domain'].value_counts().head(10)}")

# Load existing raw CSV and append new articles
raw_path = "../01_data/raw/news/gdelt_wti_raw.csv"
if os.path.exists(raw_path):
    df_existing_raw = pd.read_csv(raw_path)
    existing_urls = set(df_existing_raw['url'].dropna().tolist())
    df_truly_new = df_all[~df_all['url'].isin(existing_urls)].reset_index(drop=True)
    df_combined_raw = pd.concat([df_existing_raw, df_truly_new], ignore_index=True)
    print(f"Truly new articles (not in existing raw): {len(df_truly_new)}")
else:
    df_combined_raw = df_all
    df_truly_new = df_all
    print("No existing raw CSV found — saving all as new")

df_combined_raw.to_csv(raw_path, index=False)
print(f"Raw CSV updated: {len(df_combined_raw)} total articles")

New articles after dedup: 26075

Languages:
language
English      9029
Chinese      6690
Russian      1658
Korean        772
Ukrainian     745
Name: count, dtype: int64

Top domains:
domain
finance.eastmoney.com    523
finance.yahoo.com        406
chinatimes.com           373
baijiahao.baidu.com      372
finance.sina.com.cn      324
gold.cnfol.com           267
marketscreener.com       264
163.com                  259
udn.com                  259
mp.cnfol.com             258
Name: count, dtype: int64
Truly new articles (not in existing raw): 24397
Raw CSV updated: 76345 total articles


### Cleaning and saving to both CSV and DB

In [15]:
# Clean new articles only
df_new_clean = df_truly_new[df_truly_new['language'] == 'English'].copy()
df_new_clean = df_new_clean.drop_duplicates(subset='title').reset_index(drop=True)
df_new_clean['datetime'] = pd.to_datetime(
    df_new_clean['seendate'], format='%Y%m%dT%H%M%SZ', utc=True
)
df_new_clean = df_new_clean[['title', 'datetime', 'domain', 'url']]\
    .sort_values('datetime').reset_index(drop=True)

print(f"New clean English articles: {len(df_new_clean)}")
print(f"Range: {df_new_clean['datetime'].min()} to {df_new_clean['datetime'].max()}")

# Append to clean CSV
clean_path = "../01_data/processed/gdelt_wti_clean.csv"
if os.path.exists(clean_path):
    df_existing_clean = pd.read_csv(clean_path)
    # Dedup against existing clean CSV by URL
    existing_clean_urls = set(df_existing_clean['url'].dropna().tolist())
    df_new_clean = df_new_clean[~df_new_clean['url'].isin(existing_clean_urls)]\
        .reset_index(drop=True)
    df_combined_clean = pd.concat([df_existing_clean, df_new_clean], ignore_index=True)
else:
    df_combined_clean = df_new_clean

df_combined_clean.to_csv(clean_path, index=False)
print(f"Clean CSV updated: {len(df_combined_clean)} total articles")

# Append to DB
conn = sqlite3.connect("../01_data/wti_thesis.db")
max_id = pd.read_sql("SELECT MAX(id) as max_id FROM articles", conn)['max_id'][0]
df_new_clean['id'] = range(int(max_id) + 1, int(max_id) + 1 + len(df_new_clean))
df_new_clean['datetime'] = df_new_clean['datetime'].astype(str)
df_new_clean['body'] = None
df_new_clean['body_valid'] = None

df_new_clean[['id', 'title', 'datetime', 'domain', 'url', 'body', 'body_valid']]\
    .to_sql('articles', conn, if_exists='append', index=False)

print(f"Articles appended to DB: {len(df_new_clean)}")
print(pd.read_sql("SELECT COUNT(*) as total FROM articles", conn))
conn.close()

New clean English articles: 6469
Range: 2026-01-01 01:15:00+00:00 to 2026-05-12 00:30:00+00:00
Clean CSV updated: 22795 total articles
Articles appended to DB: 0
   total
0  22795


### Summary

In [14]:
print(f"Total articles: {len(df_all)}")
print(f"Weeks covered: {df_all['week_start'].nunique()}")
print(f"\nLanguages:\n{df_all['language'].value_counts().head()}")
print(f"\nTop domains:\n{df_all['domain'].value_counts().head(10)}")
print(df_all[['title', 'seendate', 'domain']].head(5))

Total articles: 26075
Weeks covered: 19

Languages:
language
English      9029
Chinese      6690
Russian      1658
Korean        772
Ukrainian     745
Name: count, dtype: int64

Top domains:
domain
finance.eastmoney.com    523
finance.yahoo.com        406
chinatimes.com           373
baijiahao.baidu.com      372
finance.sina.com.cn      324
gold.cnfol.com           267
marketscreener.com       264
163.com                  259
udn.com                  259
mp.cnfol.com             258
Name: count, dtype: int64
                                               title          seendate  \
0  原油交易提醒 ： IEA上调全球原油需求预期 ， 油价再次向上走强 _ 国际原油市场 _ 黄...  20260122T050000Z   
1  原油交易提醒 ： IEA上调全球原油需求预期 ， 油价再次向上走强 _ 期市动态 _ 期货 ...  20260122T050000Z   
2     主次节奏 ： 原油多头动能赚强 ， 日内走势看涨 _ 国际原油市场 _ 黄金网 _ 中金在线  20260122T021500Z   
3                  轩锋 黄金剑指5000大关 ， 原油空头保持 ！_ 中金在线财经号  20260123T033000Z   
4         主次节奏 ： 原油大幅回落维持区间内运行 _ 国际原油市场 _ 黄金网 _ 中金在线  20260123T023000Z   

              domain  
0     gold.cnfol.c

In [16]:
conn = sqlite3.connect("../01_data/wti_thesis.db")
db_urls = set(pd.read_sql("SELECT url FROM articles", conn)['url'].tolist())
clean_df = pd.read_csv("../01_data/processed/gdelt_wti_clean.csv")
clean_urls = set(clean_df['url'].dropna().tolist())

print(f"DB articles: {len(db_urls)}")
print(f"Clean CSV articles: {len(clean_df)}")
print(f"In clean CSV but not in DB: {len(clean_urls - db_urls)}")
conn.close()

DB articles: 22795
Clean CSV articles: 22795
In clean CSV but not in DB: 0
